In [0]:
orders_df = spark.table(
    "logistics.silver.shipment_orders"
)

tracking_df = spark.table(
    "logistics.silver.shipment_tracking_events"
)

orders_df.count()
tracking_df.count()

In [0]:
from pyspark.sql.functions import col

invalid_events = tracking_df.filter(
    col("shipment_id").isNull() |
    col("tracking_event_id").isNull() |
    col("event_ts").isNull() |
    col("event_type").isNull()
)

valid_events = tracking_df.filter(
    col("shipment_id").isNotNull() &
    col("tracking_event_id").isNotNull() &
    col("event_ts").isNotNull() &
    col("event_type").isNotNull()
)

print("Valid events:", valid_events.count())
print("Invalid events:", invalid_events.count())

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

dedup_window = Window.partitionBy(
    "tracking_event_id"
).orderBy(
    col("received_ts").asc_nulls_last()
)

dedup_events = (
    valid_events
    .withColumn(
        "rn",
        row_number().over(dedup_window)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
from pyspark.sql.functions import (
    col, row_number
)
from pyspark.sql.window import Window

history_window = Window.partitionBy(
    "shipment_id"
).orderBy(
    col("event_ts").asc(),
    col("received_ts").asc_nulls_last(),
    col("tracking_event_id").asc()
)

status_history = dedup_events.withColumn(
    "event_sequence",
    row_number().over(history_window)
)

In [0]:
from pyspark.sql.functions import lag

current_window = Window.partitionBy(
    "shipment_id"
).orderBy(
    col("event_ts").desc(),
    col("received_ts").desc_nulls_last(),
    col("tracking_event_id").desc()
)

# Previous event per shipment (by event_sequence), computed before keeping only the latest row
prev_window = Window.partitionBy("shipment_id").orderBy("event_sequence")

current_status = (
    status_history
    .withColumn("prev_tracking_event_id", lag("tracking_event_id").over(prev_window))
    .withColumn("prev_event_type", lag("event_type").over(prev_window))
    .withColumn("prev_event_ts", lag("event_ts").over(prev_window))
    .withColumn("prev_received_ts", lag("received_ts").over(prev_window))
    .withColumn(
        "rn",
        row_number().over(current_window)
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
from pyspark.sql.functions import when, coalesce, lit, current_timestamp

# Business columns of the shipment order (target of the CDC)
order_cols = [
    "shipment_id", "order_id", "customer_id", "carrier_id",
    "origin_warehouse_id", "destination_city", "created_ts",
    "promised_delivery_ts", "package_weight_kg", "service_level",
    "shipping_cost", "shipment_status"
]

# Tracking columns of the latest event per shipment (event_type = new status)
tracking_cols = [
    "tracking_event_id", "event_ts", "facility_id", "event_type",
    "event_city", "scan_source", "scan_quality", "received_ts"
]

# CDC metadata columns
cdc_cols = [
    "event_sequence", "prev_tracking_event_id", "prev_event_type",
    "prev_event_ts", "prev_received_ts", "updated_shipment_status",
    "cdc_operation", "status_changed", "_cdc_processed_ts"
]

latest_event = current_status.select(
    "shipment_id", *tracking_cols,
    "event_sequence", "prev_tracking_event_id", "prev_event_type",
    "prev_event_ts", "prev_received_ts"
)

# Orders drive the CDC: one row per shipment_id in orders_df (left join keeps orders without events)
# shipment_status = ORIGINAL order status (before CDC)
status_history_cdc = (
    orders_df.select(*order_cols, "_ingestion_timestamp")
    .join(latest_event, on="shipment_id", how="left")
    # Previous event fallback: when no earlier tracking event exists, the order record
    # itself (original shipment_status) is treated as the previous event
    .withColumn("prev_tracking_event_id", coalesce(col("prev_tracking_event_id"), col("order_id")))
    .withColumn("prev_event_type", coalesce(col("prev_event_type"), col("shipment_status")))
    .withColumn("prev_event_ts", coalesce(col("prev_event_ts"), col("created_ts")))
    .withColumn("prev_received_ts", coalesce(col("prev_received_ts"), col("_ingestion_timestamp")))
    .withColumn(
        "updated_shipment_status",
        coalesce(col("event_type"), col("shipment_status"))
    )
    .withColumn(
        "cdc_operation",
        when(col("event_type").isNull(), lit("NO_EVENT"))
        .when(col("shipment_status").eqNullSafe(col("event_type")), lit("NO_CHANGE"))
        .otherwise(lit("UPDATE"))
    )
    .withColumn("status_changed", col("cdc_operation") == "UPDATE")
    .withColumn("_cdc_processed_ts", current_timestamp())
    .select(*order_cols, *tracking_cols, *cdc_cols)
)

# Current table: same order columns, shipment_status replaced by the post-CDC value
shipment_current_cdc = status_history_cdc.select(
    *[c for c in order_cols if c != "shipment_status"],
    col("updated_shipment_status").alias("shipment_status")
).select(*order_cols)

print("Orders rows:", orders_df.count())
print("Status history rows:", status_history_cdc.count())
print("Current rows:", shipment_current_cdc.count())
print("History columns:", len(status_history_cdc.columns), status_history_cdc.columns)
display(status_history_cdc.groupBy("cdc_operation").count().orderBy("cdc_operation"))
display(
    status_history_cdc.select(
        "shipment_id", "shipment_status", "prev_event_type",
        "event_type", "updated_shipment_status", "cdc_operation", "event_ts"
    ).orderBy(col("event_type").isNull(), "shipment_id")
)

In [0]:
status_history_cdc.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_shipment_status_history"
    )

In [0]:
shipment_current_cdc.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        "logistics.silver.silver_shipment_current"
    )
spark.table("logistics.silver.silver_shipment_current").count()